# RiskMatch V3 — Quantitative Research Lab

One notebook for the whole project. The 27 chapter numbers are navigation anchors. Execution follows the [roadmap](../docs/ROADMAP.md), [A–Z checklist](../CHECKLIST_A_Z.md) and [prerequisite registry](../docs/TASK_DEPENDENCIES.csv).

Start with chapters 01 and 12 (mandate and 11 objectives) and 02 (Discovery), alongside data work. Validate the minimal ETF backtest (BT00, chapter 23) before advanced models. Test Explore on baselines, then improve and validate all candidates before final matching.

## Table of Contents
- [01 — Client Mandate](#chapitre-01)
- [02 — Behavioral Discovery](#chapitre-02)
- [03 — US Universe](#chapitre-03)
- [04 — Sources and Licenses](#chapitre-04)
- [05 — Point-in-Time Ingestion](#chapitre-05)
- [06 — Returns and FX](#chapitre-06)
- [07 — Descriptive Statistics](#chapitre-07)
- [08 — Return Forecasting](#chapitre-08)
- [09 — Volatility and Covariance](#chapitre-09)
- [10 — Factors and Regressions](#chapitre-10)
- [11 — Tail Risk](#chapitre-11)
- [12 — Financial Objectives](#chapitre-12)
- [13 — Baselines](#chapitre-13)
- [14 — Mean-Variance](#chapitre-14)
- [15 — Black-Litterman](#chapitre-15)
- [16 — Risk Parity and Hierarchical Methods](#chapitre-16)
- [17 — Downside Optimization](#chapitre-17)
- [18 — Advanced Optimization](#chapitre-18)
- [19 — Passive and Active Strategies](#chapitre-19)
- [20 — Alternative Strategies](#chapitre-20)
- [21 — US Treasuries](#chapitre-21)
- [22 — Futures](#chapitre-22)
- [23 — Walk-Forward Backtesting](#chapitre-23)
- [24 — Stress and Monte Carlo](#chapitre-24)
- [25 — Improvement, Matching and Comparison](#chapitre-25)
- [26 — Dynamic Management](#chapitre-26)
- [27 — Integration and Validation](#chapitre-27)

## Shared Setup
Run this cell before starting.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'riskmatch').exists())
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

### INFRA.01 — Environment validation (2026-11-24)

**Observed baseline:** macOS arm64, Python 3.12.6, pip 24.2, Node 23.11.1. The active virtual environment inherits system packages. Installed research libraries: NumPy 2.2.5, pandas 2.2.3, SciPy 1.15.2, JupyterLab 4.5.6 and nbformat 5.10.4.

**Evidence:** `pip check` reported no broken requirements; 8 Python unit tests and 5 JavaScript tests passed. A non-blocking pip cache permission warning was observed.

**Gap:** CI targets Python 3.11, the local environment uses 3.12, and inherited packages and unpinned transitive dependencies prevent repeatable clean installation.

**Decision D001 (approved by owner, 2026-11-24):** standardize on Python 3.12, isolate the virtual environment, pin all shared dependencies in `requirements.lock`, and preserve licensed data locally. CI now uses Python 3.12. WRDS, other licensed datasets, derived files, browser saves, credentials and logs stay outside Git.

**Implementation evidence:** fresh `.venv` created from the lock using INSTALL.command (the old environment is preserved in `.venv-legacy`). `pip check` passed, 13 Python tests and 5 Web tests passed, and eight explicitly synthetic notebook cells executed with the Python 3.12 kernel. The Jupyter runtime now lives under ignored `.run/`; optional SOCKS support is locked for proxy environments. The web health endpoint returned success and Jupyter returned HTTP 302. Process inspection is restricted, so STOP remains to be verified in the owner's Terminal. No UI or licensed data change. Linux CI remains to be verified remotely.

### G00 — Shared contracts and eligibility (CH27.01–02, SYS.01–02)

Prepare the versioned interfaces before financial modeling: ClientProfile, ConstraintSet, GoalSpec, Card, MarketSnapshot, ForecastBundle, RiskReport, StrategySpec, OptimizerSpec, PortfolioCandidate, Evaluation and Selection. All new contract outputs use a UTC timestamp, run ID, model and schema versions, explicit units, status, warnings and provenance. Keep forecasts separate from risk, strategy from optimizer, and funded weights from futures contracts. Detailed mandate inputs are defined in chapter 01; the legacy demonstration is not a validated financial model.

The eligibility registry covers 22 optimizers, 19 strategies, 13 factors and 10 dynamic methods. It fails closed without independent validation, necessary data, instrument verification, capacity, frequency, and required permissions or margin evidence. The 11 objectives are not methods. No method is approved merely because its interface exists. See `src/riskmatch/contracts.py`, `src/riskmatch/eligibility.py`, `tests/unit/test_contracts.py` and [architecture](../docs/ARCHITECTURE.md).

Collaboration conventions and experiment registry: [CONTRIBUTING](../CONTRIBUTING.md), [experiment registry](../docs/EXPERIMENT_REGISTRY.md). D002: public repository for now; restricted data remain local. Commit c05aaee was confirmed on origin/main and the owner reported green GitHub Actions. Both local ports stopped responding after the owner ran STOP.command. D003: solo development, direct validated pushes permitted; require collaborator access controls, PR review and main protection before collaboration. G00 closed in solo mode. Next: chapter 01 Client Mandate, together with the objective specifications in chapter 12.

In [ ]:
from datetime import datetime, timezone
from riskmatch.contracts import RunMetadata, ForecastBundle, RiskReport
from riskmatch.eligibility import REGISTRY, assess
meta = RunMetadata(as_of=datetime(2026, 11, 24, tzinfo=timezone.utc), run_id='synthetic-g00',
                   model_version='fixture-v1', units=(('return', 'decimal'),), provenance=('synthetic',))
forecast = ForecastBundle(meta=meta, snapshot_id='fixture', instrument_ids=('ETF-A',),
                          expected_returns=(0.01,), horizon_periods=1, frequency='monthly', currency='USD')
risk = RiskReport(meta=meta, snapshot_id='fixture', instrument_ids=('ETF-A',),
                  covariance=((0.002,),), frequency='monthly', currency='USD')
eligibility = assess('OPT01', instruments=frozenset({'equity_etf'}), data=REGISTRY['OPT01'].data,
                     frequency='monthly', capacity_verified=True)
assert not eligibility.eligible and 'method_not_validated' in eligibility.reasons
len(REGISTRY), forecast.meta.status, eligibility.reasons

<a id="chapitre-01"></a>

## 01 — Client Mandate

**Objective:** Define ClientProfile, hard constraints, preferences and feasibility.

Reference: [A–Z Guide, chapter 1](../docs/QUANT_GUIDE_A_Z.md).

Status: chapter 01 implementation, deterministic validation only. Full portfolio feasibility and advisory suitability are not evaluated. See [Client Mandate](../docs/CLIENT_MANDATE.md).

### 01.1 Theory and Assumptions
CH01.01–05: choose CHF/USD; cash, liquid investments, other assets, debt and reserve are distinct. Net worth = cash + investments + other assets − debt; investable wealth = cash + investments − reserve. Capital is allocated from investable wealth. All flows use the base currency; mixed-currency goals require an explicit CHF/USD scenario. Real targets = current purchasing power × (1+inflation)^T. Dates are ACT/365.25. Contributions and withdrawals have separate signs and dates; goals have objective IDs, currency, basis, priority and penalty.

CH01.06: W(T)=C(1+r)^T+Σ CF_i(1+r)^(T−t_i); solve required return by Brent root finding in log(1+r), rejecting ambiguous sign patterns and interim borrowing. Zero-return funding gaps are diagnostics, not a market forecast.

CH01.07–12: explicit loss capacity capped by capital, liquid wealth and minimum zero-return balance over the next 12 months. Stress cap = min(declared stress cap, loss budget, capacity/capital). Risk willingness and required risk remain distinct. Experience and permissions are independent. Hard limits never relax silently; missing optional limits are null. Structural conflicts are returned explicitly.

CH01.13–15: version 2.0.0 payload, synthetic fixture, API preview, before/after review and explicit confirmation. Restricted real data remain outside Git; older minimal saves retain legacy scope. The USD demo cannot evaluate CHF or unmodeled hard constraints. No model approval is implied.

In [ ]:
import json
from riskmatch.profiling.mandate import validate_mandate
profile = json.loads((root / 'configs' / 'mandate-example.json').read_text())
mandate_report = validate_mandate(profile)
assert mandate_report['status'] == 'structurally_valid'
assert mandate_report['net_worth'] == 180000 and mandate_report['investable_wealth'] == 130000
assert mandate_report['goals'][0]['zero_return_gap'] == 20000
{k: mandate_report[k] for k in ('net_worth', 'investable_wealth', 'monthly_surplus', 'effective_stress_limit', 'goals', 'combined_required_return')}

### 01.2 Validation
- [x] Analytical case and synthetic fixture (tests/unit/test_mandate.py)
- [x] Dated flows and explicit scenario inputs; no market observations
- [x] Edge cases, CHF/USD conversion and constraint sensitivity
- [ ] Out-of-sample market results are not part of mandate validation

### 01.3 Promotion to the Application
- [x] Extract functions into src/riskmatch/profiling/mandate.py
- [x] Add unit, loopback API and Web App confirmation tests
- [x] Connect API/UI and show limitations
- [ ] Owner visual confirmation and GitHub push (solo mode)

<a id="chapitre-02"></a>

## 02 — Behavioral Discovery

**Objective:** Version cards, responses, coverage, score and sensitivity.

Reference: [A–Z Guide, chapter 20](../docs/QUANT_GUIDE_A_Z.md).

Status: implemented experimental Discovery v2 (CH02.01–15 and CH02.21). Human validation CH02.16–20 remains open. See [behavioral protocol](../docs/RISKMATCH_BEHAVIORAL.md) and [draft pilot](../docs/BEHAVIORAL_PILOT.md).

### 02.1 Theory and Assumptions
Risk tolerance, loss sensitivity, path tolerance, ambiguity, time preference, style, confidence and consistency are separate constructs. Framing, anchoring and recency are research hypotheses, not individual diagnoses. The 60-card bank contains 16 lottery, 12 paired frames, 12 trajectories, 8 ambiguity/time, 8 style and 4 repeat items. Sessions select 8/6/6/4/4/2, balance placement, preserve matched pairs, store seed/version and record response locale.

Only 20 lottery/frame/trajectory items enter the exploratory score. Let k be prudent resolved answers and n resolved scored answers. Score = floor(1+29 k/n+0.5) if n/20≥0.70; otherwise unavailable. Missing sensitivity = [round(1+29k/20), round(1+29(k+20−n)/20)]. Compare canonical choices after swaps; report repeat differences and framing differences separately. The range is not a confidence interval. Confidence is optional self-report. No score changes the financial loss budget.

Utility exploration: CRRA (w^(1−γ)−1)/(1−γ), log(w) at γ=1; prospect value x^α or −λ_loss|x|^α. Objective probabilities, fixed logistic temperature and small grid; not a calibrated person-level risk parameter. Separate from score and optimizer λ.

Pilot readiness: synthetic complete anchor panel, difficulty, corrected discrimination, test-retest bootstrap, exploratory omega and eigenvalues, person-level 60/20/20 splits, logistic anchor proxy, language/order DIF and independent choice log-loss. Not human evidence or latent IRT calibration. Actual pilot, interviews, expert review and validation remain CH02.16–20.

In [ ]:
from riskmatch.behavioral.discovery import bank, session, score as discovery_score
from riskmatch.behavioral.utility import explore_likelihoods
from riskmatch.behavioral.pilot_analysis import synthetic_panel, psychometrics, logistic_readiness
discovery = session(42, 'CHF')
prudent = {c['id']: c['prudent_response'] for c in discovery['cards'] if c['scored']}
risky = {key: ('dislike' if value == 'like' else 'like') for key, value in prudent.items()}
assert len(bank()) == 60 and len(discovery['cards']) == 30
assert discovery_score(42, 'CHF', prudent)['score'] == 30
assert discovery_score(42, 'CHF', risky)['score'] == 1
assert discovery_score(42, 'CHF', dict(list(prudent.items())[:13]))['score'] is None
utility_demo = explore_likelihoods(42, 'CHF', prudent)
pilot = synthetic_panel(n=120, items=20, seed=42)
synthetic_psychometrics = psychometrics(pilot, bootstrap=50)
synthetic_logistic = logistic_readiness(pilot)
{'score': discovery_score(42, 'CHF', prudent)['score'], 'utility_status': utility_demo['status'], 'synthetic_retest': synthetic_psychometrics['test_retest'], 'synthetic_test_loss': synthetic_logistic['independent_choice_test_loss'], 'human_validation': False}

### 02.2 Validation
- [x] Bank quotas, deterministic placement, pair equivalence and extreme/missing/paired response fixtures
- [x] Synthetic person-level train/validation/test split; no response-level leakage
- [x] Score sensitivity, utility likelihood and independent synthetic baseline comparison
- [ ] Actual pilot validity, test-retest, interviews and independent review remain open

### 02.3 Promotion to the Application
- [x] Extract functions into src/riskmatch/behavioral
- [x] Unit, HTTP and web flow tests
- [x] Connect API/UI: consent, randomized session, score, separate dimensions, undo and withdrawal
- [ ] Owner visual confirmation and GitHub push (solo mode)

<a id="chapitre-03"></a>

## 03 — US Universe

**Objective:** Build the ETF security master: ticker, currency, class and dates.

**Fixed-income universe:** only T-Bills, Treasury Notes, Treasury Bonds, TIPS and bond ETFs with verified Treasury-only point-in-time holdings; exclude corporate, agency, municipal, MBS, foreign sovereign and mixed ETFs. UNIV.01 is a hard allocation filter.

Reference: [A–Z Guide, chapter 2](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 03.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 03.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 03.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-04"></a>

## 04 — Sources and Licenses

**Objective:** Catalog providers, licenses and data availability dates.

Reference: [A–Z Guide, chapter 3](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 04.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 04.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 04.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-05"></a>

## 05 — Point-in-Time Ingestion

**Objective:** Normalize snapshots, corporate actions, calendars and leakage tests.

Reference: [A–Z Guide, chapter 3.1](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 05.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 05.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 05.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-06"></a>

## 06 — Returns and FX

**Objective:** Calculate simple/log/total returns, FX and reconcile adjusted prices.

Reference: [A–Z Guide, chapter 3.1](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 06.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 06.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 06.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-07"></a>

## 07 — Descriptive Statistics

**Objective:** Means, volatility, covariance, correlation and charts.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 07.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
import pandas as pd
from riskmatch.prototype import synthetic_returns, ASSETS
r = pd.DataFrame(synthetic_returns(), columns=ASSETS)
r.describe()

### 07.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 07.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-08"></a>

## 08 — Return Forecasting

**Objective:** Historical, EWMA, CAPM, Black-Litterman and uncertainty.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 08.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 08.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 08.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-09"></a>

## 09 — Volatility and Covariance

**Objective:** EWMA, GARCH, shrinkage, DCC; PSD and OOS comparison.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 09.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 09.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 09.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-10"></a>

## 10 — Factors and Regressions

**Objective:** 13 factors, exposures, scores, OLS, PCA and attribution.

Reference: [A–Z Guide, chapter 8](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 10.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 10.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 10.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-11"></a>

## 11 — Tail Risk

**Objective:** VaR, ES, EVT, copulas, drawdown and validation.

Reference: [A–Z Guide, chapter 10](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 11.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
from riskmatch.prototype import synthetic_returns, metrics
metrics(synthetic_returns(), [.25]*4)

### 11.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 11.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-12"></a>

## 12 — Financial Objectives

**Objective:** Formalize the 11 objectives, units, constraints and loss function.

**Order:** specify GoalSpec and OBJxx.01 with the mandate (G01); evaluate goals quantitatively after their required data and simulations.

Reference: [A–Z Guide, chapter 5](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 12.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 12.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 12.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-13"></a>

## 13 — Baselines

**Objective:** Cash, 1/N, 60/40 and market benchmark.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 13.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
from riskmatch.prototype import ClientProfile, candidates
[(p['name'], p['metrics']['cagr']) for p in candidates(ClientProfile(100000, 10, 1))]

### 13.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 13.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-14"></a>

## 14 — Mean-Variance

**Objective:** Minimum variance, maximum Sharpe, maximum utility, frontier and constraints.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 14.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 14.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 14.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-15"></a>

## 15 — Black-Litterman

**Objective:** Prior, views, confidence, posterior and allocation.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 15.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 15.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 15.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-16"></a>

## 16 — Risk Parity and Hierarchical Methods

**Objective:** RP, HRP, HERC, diversification and risk contributions.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 16.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 16.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 16.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-17"></a>

## 17 — Downside Optimization

**Objective:** CVaR, Sortino, Omega, drawdown, tracking error and IR.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 17.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 17.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 17.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-18"></a>

## 18 — Advanced Optimization

**Objective:** Robust, resampled, DRO, Kelly, entropy and diagnostics.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 18.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 18.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 18.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-19"></a>

## 19 — Passive and Active Strategies

**Objective:** Passive, long-only, smart beta, core-satellite and factors.

Reference: [A–Z Guide, chapter 7](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 19.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 19.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 19.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-20"></a>

## 20 — Alternative Strategies

**Objective:** Long/short, neutral, macro, CTA, arbitrage, carry, momentum and others.

Reference: [A–Z Guide, chapter 7](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 20.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 20.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 20.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-21"></a>

## 21 — US Treasuries

**Objective:** T-Bills, Notes, Bonds, TIPS: cash flows, clean/dirty prices, duration, convexity, DV01 and curve risk.

Build nominal/real curves, slope/curvature (FAC11 Treasury Curve), breakeven and key-rate DV01. No corporate credit models.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 21.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 21.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 21.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-22"></a>

## 22 — Futures

**Objective:** Expiries, multipliers, margin, collateral and roll.

Reference: [A–Z Guide, chapter 3.1](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 22.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 22.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 22.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-23"></a>

## 23 — Walk-Forward Backtesting

**Objective:** Chronology, benchmarks, fees, delayed execution and ledger.

**Order:** BT00.01–06 (G04) before advanced strategies; ETF/cash ledger, benchmarks, reconciliation and OOS folds. Extend Treasuries/futures (G07) after chapters 21–22.

Reference: [A–Z Guide, chapter 9](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 23.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 23.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 23.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-24"></a>

## 24 — Stress and Monte Carlo

**Objective:** Scenarios, terminal wealth distribution, goals, flows and uncertainty.

Reference: [A–Z Guide, chapter 11](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 24.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 24.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 24.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-25"></a>

## 25 — Improvement, Matching and Comparison

**Objective:** Separate scientific portfolio improvement from personalized selection.

### 25.A — Scientific Improvement (CH25.15–21, IMP.01–02)
Compare variants against identical baselines; measure ablations, costs, uncertainty and robustness. Publish validated candidates only.

### 25.B — RiskMatch Explore (CH25.01–14, MATCH.01)
Filter mandate constraints, construct a Pareto ranking, allow likes/dislikes and comparisons, and confirm a paper selection. Test MVP Explore on baselines (MVP.01–04) at G05; final matching waits for IMP.02.

Reference: [A–Z Guide, chapter 12](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 25.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
from riskmatch.prototype import ClientProfile, candidates
[(p['name'], p['stress_loss']) for p in candidates(ClientProfile(100000, 10, .1))]

### 25.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 25.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-26"></a>

## 26 — Dynamic Management

**Objective:** 10 dynamic models, rebalancing, monitoring and costs.

Reference: [A–Z Guide, chapter 13](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 26.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 26.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 26.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-27"></a>

## 27 — Integration and Validation

**Objective:** Reports, exports, UI tests, security, independent validation and paper trading.

Reference: [A–Z Guide, chapter 17](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 27.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 27.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 27.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation